In [1]:
import time
from pynq.overlays.base import BaseOverlay
import socket

base = BaseOverlay("base.bit")
btns = base.btns_gpio
leds = base.leds

# Sockets

This notebook has both a client and a server functionality. One PYNQ board in the group will be the client and SENDS the message. Another PYNQ board will be the server and RECEIVES the message.

## Server

Here, we'll build the server code to LISTEN for a message from a specific PYNQ board.

When we send/receive messages, we need to pieces of information which will tell us where to send the information. First, we need the IP address of our friend. Second, we need to chose a port to listen on. For an analogy, Alice expects her friend, Bob, to deliver a package to our back door. With this information, ALICE (server ip address) can wait at the BACK DOOR (port) for BOB (client ip address) to deliver the package.

Format of the information
 ipv4 address: ###.###.###.### (no need for leading zeros if the number is less than three digits)
 port: ##### (it could be 4 or 5 digits long, but must be >1024)
 
Use the socket documentation (Section 18.1.3) to find the appropriate functions https://python.readthedocs.io/en/latest/library/socket.html


In [18]:
# Create TCP socket
# TODO:
# 1: Bind the socket to the pynq board <CLIENT-IP> at port <LISTENING-PORT>
# 2: Accept connections
# 3: Receive bytes from the connection
# 4: Print the received message

import socket
import threading
import time

stop_event = threading.Event()
SERVER = "192.168.2.99"
LISTENING_PORT = 12345

def recv_loop(ip,port,stop_event):
    print("connecting...")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((ip, port))
        s.listen(1)
        conn, addr = s.accept()
        with conn:
            print('Connected by', addr)
            while not stop_event.is_set():
                data = conn.recv(1024)
                if not data:
                    break
                print(data.decode())
                time.sleep(.01)
            conn.close()
            s.close()
            print("connection closed.")
            
# def recv_loop(i,s,e):
#     print("on")
#     while not e.is_set():
#         pass
#     print("OFF")

def btn_listener(event, mask):
    print("button listener on.")
    while True:
        time.sleep(.1)
        if btns.read() & mask:
            print("stop event request")
            event.set()
            break;
    print("button listener done.")

btnListenerThread = threading.Thread(target=btn_listener, args=(stop_event,0b0001,),daemon=True)
btnListenerThread.start()

t = threading.Thread(target=recv_loop, args=(SERVER,LISTENING_PORT,stop_event,),daemon=True)
t.start()

button listener on.
connecting...


## Client

Now, we can implement the CLIENT code. 

Back to the analogy, now we're interested in delivering a package to our friend's back door. This means BOB (client ip address) is delivering a package to ALICE (server ip address) at her BACK DOOR (port)

**Remember to start the server before running the client code**

In [19]:
import socket
SERVER = "192.168.2.99" #"192.168.8.203"
PORT = 12345

# Create TCP socket
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

import socket
import threading
import time

stop_tx_event = threading.Event()

def tx(HOST,PORT, evnt):
    # Echo client program
    print("transmitting...")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, PORT))
        s.sendall(b'Hello, world')
        data = s.recv(1024)
    print('Received', repr(data))

tt = threading.Thread(target=btn_listener, args=(stop_tx_event,0b0010,),daemon=True)
tt.start()

t = threading.Thread(target=tx, args=('192.168.2.99',12345,stop_tx_event,),daemon=True)
t.start()

button listener on.
transmitting...
Connected by ('192.168.2.99', 57318)
Hello, world


On your server, you should see the message and then the server will shutdown! When we close a socket, both the client and the server are disconnected from the port.

Instead, change the function above to send 5 messages before closing.

The pseudocode looks like this

* connect the socket
* for i in range(5)
    * msg = input("Message to send: ")
    * send the message (msg)
* close the socket

In [16]:
import socket
import time
import threading

import socket
import time
import threading

def tx_loop(server, port, stop_tx_event=None):
    message = b"Hello world!\n"
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.connect((server, port))
        print(f"Connected to {server}:{port}")

        for i in range(5):
            sock.sendall(message)
            time.sleep(1)

    except Exception as e:
        print("TX error:", e)
        time.sleep(2)

    finally:
        sock.close()
        print("Socket closed")
        
ttt = threading.Thread(target=tx_loop, args=('192.168.2.99',12345,),daemon=True)
ttt.start()

Connected to 192.168.2.99:12345
Connected by ('192.168.2.99', 50730)
Hello world!

Hello world!

Hello world!

Hello world!

Hello world!

Socket closed
connection closed.


In [11]:
import socket
import time
import signal
import sys

def run_program():
    sock_l = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock_l.bind(('0.0.0.0', 12345))
    sock_l.listen()
    print('Waiting for connection')
    conn, addr = sock_l.accept()
    print('Connected')
    with conn:
        data = conn.recv(1024)
        print(data.decode())

if __name__ == '__main__':
    original_sigint = signal.getsignal(signal.SIGINT)
    signal.signal(signal.SIGINT, exit)
    run_program()


Waiting for connection
Connected
helllllloooooooooo

